### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [1]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on ACA with GPU
# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 4096 # 8736 # 131072 # 512
# )

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    # max_completion_tokens=512
)

In [4]:
response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- coding help
- summarizing information
- tutoring and problem-solving

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be very helpful, but I can also be wrong, so it’s good to verify important information.
- I don’t automatically know private or real-time information unless you provide it or my environment explicitly gives me access to it.

You can treat me like a collaborator for thinking, writing, learning, or getting unstuck.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how to write good prompts for me

### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [6]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 20, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 20,
  "results": [
    {
      "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "Best LLM Leaderboard 2026 | AI Model Rankings, Benchmarks &amp; Pricing",
      "url": "https://onyx.app/llm-leaderboard",
      "description": "<b>The</b> definitive <b>LLM</b> leaderboard \u2014 ranking the best AI <b>models</b> including Claude, GPT, Gemini, DeepSeek, Llama, and more across coding, reasoning, math, agentic, and chat benchmarks. Compare <b>LLM

### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [7]:
import httpx
from langchain.tools import tool
from markdownify import markdownify

@tool
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

In [8]:
from IPython.display import Markdown, display

search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

last_message = None

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_dpX7mRsdd7FmTboS4Q63MQ7W)
 Call ID: call_dpX7mRsdd7FmTboS4Q63MQ7W
  Args:
    query: latest open LLM models 2026 parameters context window release date open weights 2026
    limit: 50
    searchMode: request
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest open LLM models 2026 parameters context window release date open weig

I searched 50 result pages and fetched a representative set of source webpages. Based on those sources, the strongest answer is:

## Short answer

As of **2026**, the most notable **open-weight / open-source LLMs** are led by:

- **DeepSeek V4 Pro / Flash**
- **Qwen 3.5 / 3.6 family**
- **GLM-5 / GLM-5.1**
- **Kimi K2.5 / K2.6**
- **MiMo-V2.5 / V2.5-Pro**
- **Mistral 3 family / Mistral Large 3**
- **Gemma 4**
- **Llama 4 Scout / Maverick**
- plus strong secondary options like **MiniMax-M2.7**, **Nemotron 3**, **gpt-oss-20B / 120B**, and **Phi-4-mini** for smaller deployments.

Because “latest” can mean either **newest release date** or **best currently available**, I compare the main frontier open models that are both recent and widely referenced in 2026.

---

# Comparison table

## Frontier open LLMs in 2026

| Model | Developer | Release date | Parameters | Active params | Context window | Main strengths | Main limitations |
|---|---|---:|---:|---:|---:|---|---|
| **DeepSeek V4 Pro** | DeepSeek | **2026-04-24** | **1.6T** | **49B** | **1M** | top-tier reasoning, coding, agentic tasks, long context | enormous serving cost, not practical for most local setups |
| **DeepSeek V4 Flash** | DeepSeek | **2026-04-24** | **284B** | **13B** | **1M** | much cheaper/faster than Pro, still strong for agents/coding | below Pro on hardest knowledge/reasoning tasks |
| **Qwen 3.5 397B-A17B** | Alibaba | **2026-02-16** | **397B** | **17B** | **262K** native, some sources mention extension beyond 1M | best all-round open family, strong multilingual + coding + multimodal | MoE infra complexity; very long context expensive in practice |
| **Qwen 3.6 27B** | Alibaba | **2026-04-22** | **27.8B** | **27.8B** | **262K** | dense flagship, strong coding and reasoning for its size | not frontier-class versus trillion/MoE leaders |
| **Qwen 3.6 35B-A3B** | Alibaba | **2026-04-16** | **36B** | **3B** | **262K** | very efficient coding/agentic model, high throughput | lower ceiling than larger flagships |
| **GLM-5.1** | Z AI | **2026** (latest public 2026 listing; exact launch page not fetched) | **744B** | **40B** | **200K** | excellent coding and long-horizon agentic work | smaller context than 1M-class leaders |
| **GLM-5** | Z AI | **2026-02** | **744B** | **40B** | **200K–205K** | strong reasoning/coding, MIT license | less context than DeepSeek/MiMo/Llama Scout |
| **Kimi K2.6** | Moonshot AI | **2026** | **~1T** | **32B** | **256K** | top coding and agentic orchestration, strong long-horizon tasks | modified MIT terms; shorter context than 1M models |
| **Kimi K2.5** | Moonshot AI | **2026** | **~1T** | **32B** | **256K** | very strong reasoning/coding, widely available | still operationally heavy |
| **MiMo-V2.5-Pro** | Xiaomi | **2026** | **1.02T** | **42B** | **1M** | elite coding agents, token-efficient long-context reasoning | newer ecosystem, less tooling than Qwen/Llama |
| **MiMo-V2.5** | Xiaomi | **2026** | **310B** | **15B** | **1M** | multimodal + long context + better deployability than Pro | below Pro on hardest software engineering tasks |
| **Mistral Large 3** | Mistral AI | **2025-12** | **675B** | **41B** | **256K** | strong permissive Apache 2.0 model, multilingual, multimodal | no 1M context, lower benchmark standing than top 2026 Chinese frontier models |
| **Ministral 3 14B** | Mistral AI | **2025-12** | **14B** | **14B** | **256K** | excellent edge/local model, multimodal | not frontier-level |
| **Gemma 4 31B** | Google | **2026-03-31 / public 2026-04-02** | **30.7B** | **30.7B** | **256K** | strongest dense open model in its class, easy single-H100 deployment | smaller than frontier MoE leaders; 31B variant lacks audio input |
| **Gemma 4 26B-A4B** | Google | **2026-03-31 / public 2026-04-02** | **25.2B** | **3.8B** | **256K** | very efficient MoE, good agentic/function calling support | lower ceiling than 31B dense and frontier giant MoEs |
| **Llama 4 Scout** | Meta | **2025-04** | **109B** | **17B** | **10M** | unmatched open long context | license restrictions, practical use of 10M is hardware-intensive |
| **Llama 4 Maverick** | Meta | **2025-04** | **402B** (often rounded 400B) | **17B** | **1M** | strong general/multimodal model, long context | license restrictions, not leading on 2026 coding/agentic charts |
| **MiniMax-M2.7** | MiniMax | **2026** | **230B** | **10B** | **205K** | strong productivity and agentic workflows, fast | non-commercial or modified terms depending variant |
| **gpt-oss-120B** | OpenAI | **2026** | **117B** | **5.1B** | **131K** | strong reasoning under permissive Apache 2.0 | much smaller than frontier open leaders |
| **gpt-oss-20B** | OpenAI | **2026** | **21B** | **3.6B** | **131K** | good small open reasoning model | not frontier-class quality |
| **Phi-4-mini** | Microsoft | **2025/2026 usage in 2026 guides** | **3.8B** | **3.8B** | **128K** | best low-resource local option | far below frontier systems |

---

# Model-by-model capabilities and limitations

## 1. DeepSeek V4 Pro / Flash
### Capabilities
- Among the strongest open models in 2026 for:
  - reasoning
  - coding
  - agentic workflows
  - long-context tasks
- **1M token context** is a major differentiator.
- Pro version is especially strong on hard reasoning and knowledge-heavy tasks.
- Flash version offers much better price/performance.

### Limitations
- These are still very large MoE systems.
- Self-hosting is realistic only for serious GPU infrastructure.
- Practical long-context use remains expensive even if the nominal window is 1M.

### Best for
- enterprise agent systems
- coding assistants at scale
- long-document/RAG workflows
- research on long context

---

## 2. Qwen 3.5 / 3.6
### Capabilities
- Probably the **best balanced open family** in 2026.
- Strong across:
  - coding
  - multilingual tasks
  - reasoning
  - agents
  - multimodal
- Apache 2.0 licensing is a major advantage.
- The 27B dense and 35B-A3B variants are more deployable than trillion-class models.

### Limitations
- The flagship 397B-A17B still needs real infra.
- MoE deployment is operationally more complex than dense models.
- Claimed long-context extensions beyond native window may not be equally reliable in all stacks.

### Best for
- teams wanting strong open models with permissive licensing
- multilingual products
- private coding assistants
- balanced cost/performance deployment

---

## 3. GLM-5 / GLM-5.1
### Capabilities
- Excellent coding and software engineering performance.
- Strong on long-horizon agentic tasks.
- MIT license is very favorable.
- 744B total / 40B active gives strong quality without full dense compute cost.

### Limitations
- Context is around **200K**, which is good but behind 1M-class leaders.
- Ecosystem/tooling visibility is lower than Qwen/Llama/Mistral.

### Best for
- coding agents
- long-running workflows
- enterprises that want permissive licensing

---

## 4. Kimi K2.5 / K2.6
### Capabilities
- Frequently cited among the **top open coding and agentic models** in 2026.
- Strong long-horizon task execution.
- Good multimodality and preserved reasoning traces in some variants.

### Limitations
- License is **modified MIT**, not plain MIT.
- Context window is **256K**, not 1M.
- Very large model; not broadly local-friendly.

### Best for
- autonomous coding agents
- complex software workflows
- high-end open model deployments

---

## 5. MiMo-V2.5 / Pro
### Capabilities
- Xiaomi’s family is one of the most impressive 2026 entrants.
- **1M context**
- strong coding
- strong long-context reasoning
- multimodal support in V2.5
- token-efficient behavior is repeatedly highlighted

### Limitations
- Smaller ecosystem and community than Qwen/Llama
- Less battle-tested in mainstream open tooling stacks

### Best for
- long-context coding agents
- multimodal agent systems
- orgs comfortable adopting newer model ecosystems

---

## 6. Mistral 3 / Mistral Large 3
### Capabilities
- Strong permissive **Apache 2.0** release.
- Good multilingual and multimodal support.
- Better openness and deployment friendliness than many rivals.
- Ministral 3 models are attractive for local/edge.

### Limitations
- Large 3 is no longer the top raw performer against leading 2026 Chinese frontier models.
- Context is **256K**, not 1M.
- More “strong and practical” than “absolute benchmark leader.”

### Best for
- enterprises wanting a permissive and mature open ecosystem
- multilingual/multimodal workloads
- smaller local models via Ministral

---

## 7. Gemma 4
### Capabilities
- Very strong dense open models for their size.
- **31B** is one of the best deployable dense open models in 2026.
- 256K context is excellent for a model in this class.
- Good reasoning, coding, function calling, multilingual capability.

### Limitations
- Still smaller than the frontier MoE giants.
- Not the best choice if you need absolute top-end coding-agent performance.
- Licensing/open-weight framing should still be reviewed carefully despite broad usability.

### Best for
- single-GPU or modest enterprise deployment
- strong dense open model use
- practical multimodal apps

---

## 8. Llama 4 Scout / Maverick
### Capabilities
- **Scout’s 10M context** is the headline feature and still unique.
- Maverick offers 1M context and good multimodal capability.
- Broad tooling ecosystem due to Meta/Llama adoption.

### Limitations
- Community license restrictions matter.
- EU and scale-related terms can complicate commercial use.
- In 2026 leaderboards, Llama 4 is less dominant for coding/agentic tasks than Kimi/DeepSeek/Qwen/GLM class models.
- Huge nominal context doesn’t mean cheap or easy practical inference.

### Best for
- long-context experimentation
- large document workflows
- teams already invested in Llama tooling

---

# What are the “latest” open models by release date?

Among the main models surfaced from fetched sources:

- **DeepSeek V4** — **2026-04-24**
- **Qwen 3.6-27B** — **2026-04-22**
- **Qwen 3.6-35B-A3B** — **2026-04-16**
- **Gemma 4** — **2026-03-31** launch / **2026-04-02** public announcement
- **Qwen 3.5 family** — **2026-02-16** initial flagship release
- **GLM-5** — **2026-02**
- **Mistral Small 4** appears in comparison sources as **2026-03**, but your request asked for latest open LLMs generally; still, I focused on the larger flagship families
- **Mistral Large 3 / Ministral 3** — **2025-12**
- **Llama 4 Scout/Maverick** — **2025-04**

So if you mean **most recent frontier open releases in 2026**, the key names are:
1. **DeepSeek V4**
2. **Qwen 3.6**
3. **Gemma 4**
4. **GLM-5 / 5.1**
5. **MiMo-V2.5**
6. **Kimi K2.6**
7. **MiniMax-M2.7**

---

# Best model by use case

## Best overall frontier open model
**DeepSeek V4 Pro** or **Qwen 3.5 397B-A17B**

- **DeepSeek V4 Pro** if you want raw frontier-level reasoning/coding + 1M context.
- **Qwen 3.5** if you want the most balanced family with strong licensing and ecosystem support.

## Best coding / software engineering
- **GLM-5.1**
- **Kimi K2.6**
- **DeepSeek V4 Pro**
- **MiMo-V2.5-Pro**

## Best long context
- **Llama 4 Scout** for maximum nominal context (**10M**)
- **DeepSeek V4 / MiMo-V2.5** for more practical **1M** frontier usage

## Best deployable dense model
- **Gemma 4 31B**
- **Qwen 3.6 27B**

## Best open license / enterprise-friendliness
- **Qwen 3.5 / 3.6** — Apache 2.0
- **Mistral 3** — Apache 2.0
- **DeepSeek V4** — MIT
- **GLM-5** — MIT

## Best for local / small hardware
- **Phi-4-mini**
- **Gemma 4 E2B / E4B**
- **Ministral 3 3B/8B/14B**
- **Qwen smaller variants**

---

# Important caveats

1. **Open-source vs open-weight**
   - Many sources use these terms loosely.
   - Strictly speaking, many are **open-weight**, not fully open-source in the OSI sense.

2. **Parameter count is not everything**
   - MoE models may have massive total params but only a smaller active subset per token.
   - Example: **DeepSeek V4 Pro = 1.6T total, 49B active**.

3. **Advertised context windows are not equal**
   - A 1M or 10M context claim does not guarantee equally good quality at full length.
   - Real performance depends on serving stack, KV cache constraints, and benchmarked long-context behavior.

4. **Licensing matters**
   - Apache 2.0 / MIT are safest.
   - Llama and some others have additional restrictions.

---

# Bottom line

If I had to summarize the 2026 open LLM landscape in one sentence:

- **DeepSeek V4, Qwen 3.5/3.6, GLM-5.1, Kimi K2.6, and MiMo-V2.5-Pro** are the most important frontier open models in 2026,
- while **Gemma 4**, **Mistral 3**, and **Llama 4** remain highly relevant because they are easier to deploy, better tooled, or uniquely strong in dense-model or ultra-long-context scenarios.

If you want, I can next turn this into a:
1. **clean CSV-style table**, or
2. **ranking by best coding / reasoning / local deployment / licensing**, or
3. **hardware-oriented guide: which of these can realistically run on 24GB, 48GB, 80GB VRAM, etc.**

## References

I searched 50 web results and fetched pages including these sources used in the synthesis:

1. Mistral AI, **“Introducing Mistral 3”**  
   https://mistral.ai/news/mistral-3

2. QwenLM GitHub, **“Qwen3.6”**  
   https://github.com/QwenLM/Qwen3.6

3. Artificial Analysis, **“Comparison of Open Source Models”**  
   https://artificialanalysis.ai/models/open-source

4. ComputingForGeeks, **“Open Source LLM Comparison Table (2026)”**  
   https://computingforgeeks.com/open-source-llm-comparison/

5. BentoML, **“The Best Open-Source LLMs in 2026”**  
   https://www.bentoml.com/blog/navigating-the-world-of-open-source-large-language-models

6. NetApp Instaclustr, **“Top 7 open source LLMs for 2026”**  
   https://www.instaclustr.com/education/open-source-ai/top-7-open-source-llms-for-2026/

7. Hugging Face community article, **“The Best Open Source and Open-Weight LLM Models to Run Locally in 2026”**  
   https://huggingface.co/blog/daya-shankar/open-source-llm-models-to-run-locally

8. Vellum, **“Open Source LLM Leaderboard 2026”**  
   https://www.vellum.ai/open-llm-leaderboard

9. WhatLLM, **“Best Open Source LLM 2026 Ranking”**  
   https://whatllm.org/best-open-source-llm

10. Search result set used to identify candidate pages  
   Startpage + DuckDuckGo search over 50 results on “latest open LLM models 2026 parameters context window release date open weights”.

## Working with Sub-Agents

A main agent coordinates subagents as tools. All routing passes through the main agent.

In the subagents architecture, a central main agent (often referred to as a supervisor) coordinates subagents by calling them as tools. The main agent decides which subagent to invoke, what input to provide, and how to combine results. Subagents are stateless—they don’t remember past interactions, with all conversation memory maintained by the main agent. This provides context isolation: each subagent invocation works in a clean context window, preventing context bloat in the main conversation.

![](./images/sub_agents.png)


### When to use

Use the subagents pattern when you have multiple distinct domains (e.g., calendar, email, CRM, database), subagents don’t need to converse directly with users, or you want centralized workflow control. For simpler cases with just a few tools, use a single agent.

### Performance comparison

Different patterns have different performance characteristics. Understanding these tradeoffs helps you choose the right pattern for your latency and cost requirements.
Key metrics:
Model calls: Number of LLM invocations. More calls = higher latency (especially if sequential) and higher per-request API costs.
Tokens processed: Total context window usage across all calls. More tokens = higher processing costs and potential context limits.

In [15]:
from langchain.tools import tool
from langchain.agents import create_agent

# Create a subagent
subagent = create_agent(model=model, tools=[search_tool, fetch_webpage_content])
# subagent = create_agent(model="google_genai:gemini-3.1-pro-preview", tools=[...])

# Wrap it as an async tool so it stays in the async execution path
@tool("research", description="Research a topic and return findings")
async def call_sub_agent(query: str):
    result = await subagent.ainvoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# Main agent with subagent as a tool
main_agent = create_agent(model=model, 
                          tools=[call_sub_agent],     
                          system_prompt=(""""
    You coordinate specialized sub-agent specialized in web search. 
    Use the task tool to delegate work.
    Delegate fetch web pages to the sub agent tool and let it resume the content of that page and report it back to the main agent.
"""))


In [16]:
async for step in main_agent.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  research (call_xI0bnU35cKsBW2ZKiHaeBZXp)
 Call ID: call_xI0bnU35cKsBW2ZKiHaeBZXp
  Args:
    query: latest open LLM models 2026 open weights release parameters context window release date 50 results
  research (call_wF5QC842Z0BUwEWIqGIJevTc)
 Call ID: call_wF5QC842Z0BUwEWIqGIJevTc
  Args:
    query: 2026 open-source large language models parameters context window release date comparison open weights
  research (call_iH2bxfG6VD4ryw762WF9oust)
 Call ID: call_iH2bxfG6VD4ryw